# 🧬 Graph Neural Network for Drug Response Prediction
Predicting LN_IC50 using GNN on cell line-drug interactions

## 1. Install Required Libraries

In [1]:
# Run this cell first to install required packages
%pip install torch torch-geometric pandas scikit-learn scipy

   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.5/241.4 MB 441.3 kB/s eta 0:09:06
   ---------------------------------------- 0.5/241.4 MB 441.3 kB/s eta 0:09:06
   ---------------------------------------- 0.5/241.4 MB 441.3 kB/s eta 0:09:06
   ---------------------------------------- 0.8/241.4 MB 430.1 kB/s eta 0:09:20
   ---------------------------------------- 0.8/241.4 MB 430.1 kB/s eta 0:09:20
   ---------------------------------------- 0.8/241.4 MB 430.1 kB/s eta 0:09:20
   ---------------------------------------- 0.8/241.4 MB 430.1 kB/s eta 0:09:20
   -----------

  You can safely remove it manually.


## 2. Import Libraries

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

✅ Libraries imported successfully
PyTorch version: 2.8.0+cpu
Device: CPU


## 3. Load and Prepare Data

In [3]:
# Load data
merged_data = pd.read_csv(r"C:\Users\vinay\Downloads\merged_data.csv")

print(f"Data shape: {merged_data.shape}")
print(f"\nColumns: {merged_data.columns.tolist()[:10]}...")
print(f"\nTarget (LN_IC50) stats:")
print(merged_data['LN_IC50'].describe())

Data shape: (166644, 32)

Columns: ['COSMIC_ID', 'CELL_LINE_NAME', 'TCGA_DESC', 'DRUG_ID', 'DRUG_NAME', 'LN_IC50', 'AUC', 'Z_SCORE', 'GDSC Tissue descriptor 1', 'GDSC Tissue descriptor 2']...

Target (LN_IC50) stats:
count    166644.000000
mean          2.841726
std           2.825374
min          -8.642551
25%           1.505524
50%           3.292250
75%           4.777961
max          13.820189
Name: LN_IC50, dtype: float64


## 4. Encode Categorical Features

In [4]:
# Encode cell lines and drugs
cell_encoder = LabelEncoder()
drug_encoder = LabelEncoder()

merged_data['CELL_ID'] = cell_encoder.fit_transform(merged_data['TCGA_DESC'])
merged_data['DRUG_ID'] = drug_encoder.fit_transform(merged_data['DRUG_NAME']) if 'DRUG_NAME' in merged_data.columns else 0

n_cells = merged_data['CELL_ID'].nunique()
n_drugs = merged_data['DRUG_ID'].nunique()

print(f"Number of unique cell lines: {n_cells}")
print(f"Number of unique drugs: {n_drugs}")

Number of unique cell lines: 31
Number of unique drugs: 249


## 5. Split Data (60/20/20)

In [5]:
target = merged_data['LN_IC50']
stratify_col = merged_data['TCGA_DESC']

# Split 1: 60% train, 40% temp
split1 = StratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
for train_idx, temp_idx in split1.split(merged_data, stratify_col):
    train_data = merged_data.iloc[train_idx]
    temp_data = merged_data.iloc[temp_idx]
    stratify_temp = stratify_col.iloc[temp_idx]

# Split 2: 20% val, 20% test
split2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for val_idx, test_idx in split2.split(temp_data, stratify_temp):
    val_data = temp_data.iloc[val_idx]
    test_data = temp_data.iloc[test_idx]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Train: 99986, Val: 33329, Test: 33329


## 6. Create Graph Data Structure

In [6]:
# Select numeric features
feature_cols = merged_data.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in feature_cols if col not in ['LN_IC50', 'CELL_ID', 'DRUG_ID']]

print(f"Number of features: {len(feature_cols)}")

# Scale features
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_data[feature_cols])
val_features_scaled = scaler.transform(val_data[feature_cols])
test_features_scaled = scaler.transform(test_data[feature_cols])

def create_graph_data(data, features_scaled):
    """Convert dataframe to PyG graph data"""
    graphs = []
    
    for idx, (_, row) in enumerate(data.iterrows()):
        # Node features: [cell_features, drug_features]
        x = torch.tensor(features_scaled[idx], dtype=torch.float).unsqueeze(0)
        
        # Edge: cell -> drug connection
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)  # Self-loop for simplicity
        
        # Target
        y = torch.tensor([row['LN_IC50']], dtype=torch.float)
        
        # Additional node info
        cell_id = torch.tensor([row['CELL_ID']], dtype=torch.long)
        drug_id = torch.tensor([row['DRUG_ID']], dtype=torch.long)
        
        graph = Data(x=x, edge_index=edge_index, y=y, cell_id=cell_id, drug_id=drug_id)
        graphs.append(graph)
    
    return graphs

train_graphs = create_graph_data(train_data, train_features_scaled)
val_graphs = create_graph_data(val_data, val_features_scaled)
test_graphs = create_graph_data(test_data, test_features_scaled)

print(f"✅ Created {len(train_graphs)} training graphs")

Number of features: 4
✅ Created 99986 training graphs


## 7. Create DataLoaders

In [7]:
batch_size = 64

train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 1563
Val batches: 521
Test batches: 521


## 8. Define GNN Model

In [8]:
class DrugResponseGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(DrugResponseGNN, self).__init__()
        
        # Graph Convolutional Layers
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim // 2)
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(hidden_dim // 2, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # GCN layers
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        
        # Global pooling
        x = global_mean_pool(x, batch)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        
        return x.squeeze()

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DrugResponseGNN(input_dim=len(feature_cols), hidden_dim=128).to(device)

print("\n📊 Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")


📊 Model Architecture:
DrugResponseGNN(
  (conv1): GCNConv(4, 128)
  (conv2): GCNConv(128, 128)
  (conv3): GCNConv(128, 64)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

Total parameters: 32193


## 9. Training Setup

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(out.cpu().numpy())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson, _ = pearsonr(y_true, y_pred)
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Pearson': pearson}

print("✅ Training setup complete")

✅ Training setup complete


## 10. Train Model

In [10]:
epochs = 100
best_val_loss = float('inf')
patience = 20
patience_counter = 0

history = {'train_loss': [], 'val_loss': [], 'val_rmse': []}

print("\n🚀 Training GNN...")
print("="*60)

for epoch in range(epochs):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Validate
    val_metrics = evaluate(model, val_loader, device)
    val_loss = val_metrics['RMSE'] ** 2  # Convert RMSE to MSE
    
    # Scheduler step
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_rmse'].append(val_metrics['RMSE'])
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | "
              f"Val RMSE: {val_metrics['RMSE']:.4f} | Val R²: {val_metrics['R2']:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}")
            break

# Restore best model
model.load_state_dict(best_model_state)
print("\n✅ Training complete!")


🚀 Training GNN...
Epoch 005 | Train Loss: 2.8891 | Val RMSE: 1.7786 | Val R²: 0.5992
Epoch 010 | Train Loss: 2.8350 | Val RMSE: 1.7437 | Val R²: 0.6148
Epoch 015 | Train Loss: 2.7952 | Val RMSE: 1.7099 | Val R²: 0.6296
Epoch 020 | Train Loss: 2.7881 | Val RMSE: 1.7159 | Val R²: 0.6270

⏹️ Early stopping at epoch 21

✅ Training complete!


## 11. Final Evaluation

In [11]:
print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)

train_metrics = evaluate(model, train_loader, device)
val_metrics = evaluate(model, val_loader, device)
test_metrics = evaluate(model, test_loader, device)

print("\nTrain Metrics:")
for key, val in train_metrics.items():
    print(f"  {key}: {val:.4f}")

print("\nValidation Metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")

print("\nTest Metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

# Summary DataFrame
results_df = pd.DataFrame({
    'Set': ['Train', 'Validation', 'Test'],
    'RMSE': [train_metrics['RMSE'], val_metrics['RMSE'], test_metrics['RMSE']],
    'MAE': [train_metrics['MAE'], val_metrics['MAE'], test_metrics['MAE']],
    'R²': [train_metrics['R2'], val_metrics['R2'], test_metrics['R2']],
    'Pearson': [train_metrics['Pearson'], val_metrics['Pearson'], test_metrics['Pearson']]
})

print("\n", results_df.to_string(index=False))


FINAL EVALUATION

Train Metrics:
  RMSE: 1.7101
  MAE: 1.1671
  R2: 0.6333
  Pearson: 0.8147

Validation Metrics:
  RMSE: 1.7128
  MAE: 1.1659
  R2: 0.6284
  Pearson: 0.8105

Test Metrics:
  RMSE: 1.7224
  MAE: 1.1782
  R2: 0.6334
  Pearson: 0.8142

        Set     RMSE      MAE       R²  Pearson
     Train 1.710113 1.167145 0.633324 0.814655
Validation 1.712823 1.165922 0.628352 0.810504
      Test 1.722396 1.178163 0.633394 0.814220


## 12. Save Model and Artifacts

In [14]:
save_dir = r"C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models"
os.makedirs(save_dir, exist_ok=True)

print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

# Save PyTorch model
model_path = os.path.join(save_dir, 'gnn_model.pth')
torch.save(model.state_dict(), model_path)
print(f"✅ GNN model saved: {model_path}")

# Save complete model (with architecture)
full_model_path = os.path.join(save_dir, 'gnn_model_full.pth')
torch.save(model, full_model_path)
print(f"✅ GNN full model saved: {full_model_path}")

# Save scaler
scaler_path = os.path.join(save_dir, 'gnn_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Scaler saved: {scaler_path}")

# Save encoders
encoders_path = os.path.join(save_dir, 'gnn_encoders.pkl')
with open(encoders_path, 'wb') as f:
    pickle.dump({'cell_encoder': cell_encoder, 'drug_encoder': drug_encoder}, f)
print(f"✅ Encoders saved: {encoders_path}")

# Save metadata
metadata = {
    'model_type': 'Graph Neural Network (GNN)',
    'architecture': 'GCN with 3 graph conv layers',
    'feature_columns': feature_cols,
    'n_features': len(feature_cols),
    'hidden_dim': 128,
    'n_cells': n_cells,
    'n_drugs': n_drugs,
    'training_samples': len(train_data),
    'val_samples': len(val_data),
    'test_samples': len(test_data),
    'metrics': {
        'train': train_metrics,
        'validation': val_metrics,
        'test': test_metrics
    },
    'training_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'epochs_trained': epoch + 1
}

metadata_path = os.path.join(save_dir, 'gnn_metadata.pkl')
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✅ Metadata saved: {metadata_path}")

# Save training history
history_path = os.path.join(save_dir, 'gnn_training_history.pkl')
with open(history_path, 'wb') as f:
    pickle.dump(history, f)
print(f"✅ Training history saved: {history_path}")

print("\n" + "="*60)
print(f"ALL FILES SAVED TO: {save_dir}")
print("="*60)


SAVING MODEL
✅ GNN model saved: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models\gnn_model.pth
✅ GNN full model saved: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models\gnn_model_full.pth
✅ Scaler saved: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models\gnn_scaler.pkl
✅ Encoders saved: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models\gnn_encoders.pkl
✅ Metadata saved: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models\gnn_metadata.pkl
✅ Training history saved: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models\gnn_training_history.pkl

ALL FILES SAVED TO: C:\Users\vinay\OneDrive\Desktop\hackathon\autopharma\trained_models\gnn_models


## 13. How to Load and Use the Model

In [13]:
# Example code to load and use the model
print("""
TO LOAD AND USE THE GNN MODEL:
================================

import torch
import pickle

# Load model
model = torch.load('gnn_model_full.pth')
model.eval()

# OR load state dict
# model = DrugResponseGNN(input_dim=n_features)
# model.load_state_dict(torch.load('gnn_model.pth'))

# Load scaler
with open('gnn_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Load metadata
with open('gnn_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

# Make predictions
# X_new_scaled = scaler.transform(X_new)
# predictions = model(graph_data)
""")


TO LOAD AND USE THE GNN MODEL:

import torch
import pickle

# Load model
model = torch.load('gnn_model_full.pth')
model.eval()

# OR load state dict
# model = DrugResponseGNN(input_dim=n_features)
# model.load_state_dict(torch.load('gnn_model.pth'))

# Load scaler
with open('gnn_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Load metadata
with open('gnn_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

# Make predictions
# X_new_scaled = scaler.transform(X_new)
# predictions = model(graph_data)



## ✅ Done!
Your GNN model has been trained and saved successfully. All files are in: `C:\Users\vinay\Downloads\trained_models`